In [0]:
%pip install hana_ml
%pip install matplotlib

In [0]:
%restart_python

In [0]:
%sh curl ifconfig.me

In [0]:
import hana_ml.dataframe as dataframe
conn = dataframe.ConnectionContext(address='dd7e29f9-20d8-46e9-b28d-137228ce82be.hana.prod-us10.hanacloud.ondemand.com',
                                   port=443,
                                   user='DWC#ML',
                                   password='A8Hzb2!HFY>$;q6D|z#Ox>!f]N3/$W<u')
conn.connection.isconnected()
 
#Data exploration
# Point to the table that contains the data

# Note : Greater than 2 years of data is expected
df_remote = (conn.table('V_ML_OP', schema='DWC').filter("MONTH < '2023-03-01'"))
 
# Verify the column types
 
df_remote.select_statement
df_remote.dtypes()
 
# Retrieve a few rows of data
 
#df_remote.head(5).collect()


In [0]:
# Aggregate the tourism revenue by month
 
df_rem_agg = df_remote.agg([('sum', 'TOURISM_REVENUE', 'TOURISM_REVENUE_SUM')], group_by='MONTH')
df_rem_agg = df_rem_agg.sort('MONTH')
df_rem_agg.head(5).collect()

In [0]:
df_rem_agg = df_rem_agg.cast(['TOURISM_REVENUE_SUM'], 'INTEGER')
df_check = df_rem_agg.collect()
print(df_check.dtypes)



In [0]:
from hana_ml.algorithms.pal.tsa.additive_model_forecast import AdditiveModelForecast
amf = AdditiveModelForecast()
amf.fit(data=df_rem_agg)

# Explore the trained model in a graphical plot
 
from hana_ml.visualizers.unified_report import UnifiedReport
#UnifiedReport(amf).build().display()

In [0]:
# Create a hana_ml DataFrame that contains the dates for which we want to create a forecast.
# Start by identifying the most recent month from the training history

str_lastdate = df_rem_agg.tail(1, ref_col='MONTH').collect().iloc[0,0]
str_lastdate = str(str_lastdate)[0:10]

# Create a hana_ml DataFrame that contains the 1st day of the 12 following months
 
from hana_ml.algorithms.pal.random import binomial
months_to_forecast=12
df_rem_future = binomial(conn, n=1, p=1, num_random=months_to_forecast)
df_rem_future = df_rem_future.select('*', (f'''ADD_MONTHS(TO_DATE ('{str_lastdate}', 'YYYY-MM-DD'), ID+1)''', 'MONTH') )
df_rem_future = df_rem_future.select('MONTH', ('0', 'TARGET'))
#df_rem_future.head(20).collect()


In [0]:
# Apply the trained time-series model to create forecasts for these dates
 
df_rem_predicted = amf.predict(data=df_rem_future)
df_rem_predicted.head(5).collect()


In [0]:
# See the forecast in the graphical plot
 
from hana_ml.visualizers.unified_report import UnifiedReport
#UnifiedReport(amf).build().display()


In [0]:
# Combine the historical and predicted data in a single hana_ml data Frame
 
df_rem_predicted = df_rem_predicted.select('MONTH',
                                          ('NULL', 'TOURISM_REVENUE_SUM'),
                                          ('YHAT', 'FORECAST'),
                                          ('YHAT_LOWER', 'FORECAST_LOWER'),
                                          ('YHAT_UPPER', 'FORECAST_UPPER'))
df_rem_agg = df_rem_agg.select('*', 
                               ('NULL', 'FORECAST'),
                               ('NULL', 'FORECAST_LOWER'),
                               ('NULL', 'FORECAST_UPPER'))
df_rem_all = df_rem_predicted.union(df_rem_agg)
df_rem_all.sort('MONTH').tail(5).collect()


In [0]:
df_rem_all.save('TOURISM_REVENUE_TOTAL', force=True)